# Task 1

In [88]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\HP\.cache\kagglehub\datasets\manishkr1754\cardekho-used-car-data\versions\2


In [89]:
import pandas as pd
df=pd.read_csv(f"{path}/cardekho_dataset.csv")
print(df.head())

   Unnamed: 0       car_name    brand     model  vehicle_age  km_driven  \
0           0    Maruti Alto   Maruti      Alto            9     120000   
1           1  Hyundai Grand  Hyundai     Grand            5      20000   
2           2    Hyundai i20  Hyundai       i20           11      60000   
3           3    Maruti Alto   Maruti      Alto            9      37000   
4           4  Ford Ecosport     Ford  Ecosport            6      30000   

  seller_type fuel_type transmission_type  mileage  engine  max_power  seats  \
0  Individual    Petrol            Manual    19.70     796      46.30      5   
1  Individual    Petrol            Manual    18.90    1197      82.00      5   
2  Individual    Petrol            Manual    17.00    1197      80.00      5   
3  Individual    Petrol            Manual    20.92     998      67.10      5   
4      Dealer    Diesel            Manual    22.77    1498      98.59      5   

   selling_price  
0         120000  
1         550000  
2         2

In [90]:
df.shape

(15411, 14)

In [91]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB


In [92]:
df.isnull().sum()*100/len(df)

Unnamed: 0           0.0
car_name             0.0
brand                0.0
model                0.0
vehicle_age          0.0
km_driven            0.0
seller_type          0.0
fuel_type            0.0
transmission_type    0.0
mileage              0.0
engine               0.0
max_power            0.0
seats                0.0
selling_price        0.0
dtype: float64

### Dropping rows where selling_price is null

In [93]:
df.dropna(subset=["selling_price"], inplace=True)

In [94]:
df["mileage"]=df["mileage"].astype(str).str.replace("kmpl", "").str.replace("km", "").str.replace(",", "").str.strip()
df["mileage"]=pd.to_numeric(df["mileage"], errors="coerce")
df["engine"]=df["engine"].astype(str).str.replace("CC", "").str.replace(",", "").str.strip()
df["engine"]=pd.to_numeric(df["engine"], errors="coerce")
df["max_power"]=df["max_power"].astype(str).str.replace("bhp", "").str.replace(",", "").str.strip()
df["max_power"]=pd.to_numeric(df["max_power"], errors="coerce")

### Or regex version even better

In [95]:
for col in ["mileage", "engine", "max_power"]:
    df[col]=pd.to_numeric(
        df[col].astype(str).str.extract(r"(\d+\.?\d*)")[0],
        errors="coerce"
    )

In [96]:
for col in ["mileage", "engine", "max_power"]:
    df[col]=df[col].fillna(df[col].median())

In [97]:
df=df[df["selling_price"]!=999999999]
df=df[df["selling_price"]>=10000]

In [98]:
df=df.drop_duplicates(keep=False)

In [99]:
df.shape

(15411, 14)

# Task 2

In [100]:
df["transmission_type"]=df["transmission_type"].map({"Manual": 0, "Automatic": 1})

In [101]:
df=pd.get_dummies(df,columns=["fuel_type", "seller_type"], drop_first=True)

In [102]:
df.dtypes

Unnamed: 0                        int64
car_name                         object
brand                            object
model                            object
vehicle_age                       int64
km_driven                         int64
transmission_type                 int64
mileage                         float64
engine                            int64
max_power                       float64
seats                             int64
selling_price                     int64
fuel_type_Diesel                   bool
fuel_type_Electric                 bool
fuel_type_LPG                      bool
fuel_type_Petrol                   bool
seller_type_Individual             bool
seller_type_Trustmark Dealer       bool
dtype: object

In [103]:
df.columns.to_list()

['Unnamed: 0',
 'car_name',
 'brand',
 'model',
 'vehicle_age',
 'km_driven',
 'transmission_type',
 'mileage',
 'engine',
 'max_power',
 'seats',
 'selling_price',
 'fuel_type_Diesel',
 'fuel_type_Electric',
 'fuel_type_LPG',
 'fuel_type_Petrol',
 'seller_type_Individual',
 'seller_type_Trustmark Dealer']

### Why `drop_first=True` is used?
- It removes one dummpy column to avoid multicollinearity, where columns become linearly independent
### What happens after dropping?
- The dropped category becomes the referece against which other categories are compared
### What does a row of all zeros mean?
- It represents the dropped category

# Task 3

In [109]:
X=df.drop(columns=["selling_price"])
y=df["selling_price"]

In [110]:
X.shape

(15411, 17)

In [111]:
X.dtypes

Unnamed: 0                        int64
car_name                         object
brand                            object
model                            object
vehicle_age                       int64
km_driven                         int64
transmission_type                 int64
mileage                         float64
engine                            int64
max_power                       float64
seats                             int64
fuel_type_Diesel                   bool
fuel_type_Electric                 bool
fuel_type_LPG                      bool
fuel_type_Petrol                   bool
seller_type_Individual             bool
seller_type_Trustmark Dealer       bool
dtype: object

In [112]:
X = X.select_dtypes(include=["number", "bool"])

In [113]:
X.shape

(15411, 14)

In [114]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [116]:
baseline_pred=y_train.mean()
y_pred_baseline= [baseline_pred] * len(y_test)

In [ ]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, y_pred_baseline)
print(f"BaseLine MAE: ₹{round(mae)}")